# FlyLM · MaleCNS 8,192 · 한국어 입력→답변 학습 · Colab T4

**화자1 발화를 입력하고 화자2 답변을 예측한다. Loss는 화자2 답변과 EOS에만 적용한다.**
MaleCNS의 고정 연결, synapse count 초기화, 전달물질 부호 제약, 뉴런당 스칼라 상태는 기존 모델을 유지한다.
UTF-8 byte vocabulary를 사용하며 tokenizer를 별도로 학습하지 않는다.

Drive의 일상대화_데이터.zip에서 document[].utterance[].form을 읽는다.
문서에서 처음 등장한 화자를 화자1, 다음 화자를 화자2로 재번호화한다.
같은 화자의 연속 발화는 합치고, 연속된 화자1→화자2 turn을 하나의 입력–답변 쌍으로 만든다.
각 쌍은 직전 화자1 turn만 입력으로 쓰며 이전 대화 전체를 누적하지 않는다.
독백/3인 이상 대화/화자 ID 누락 문서는 제외하고 개수를 출력한다.
샘플 SARW2400000010.json은 독백이므로 이 학습 조건에서는 제외된다.

**Colab T4에서 위에서부터 실행한다.** Drive 인증 후 ZIP을 VM 로컬에 복사하고 uint16 토큰 캐시를 만든다.
같은 대화에서 나온 모든 쌍을 한 split에 배치하며, 정규화한 동일 입력–답변 쌍은 전체에서 중복 제거한다.
기본값은 1 epoch 안에서 최대 **1,000 optimizer updates**이다.
전체 train split 1회 학습은 첫 설정 셀에서 train_updates=None, epochs=1로 바꾼다.
결과와 best/latest 체크포인트는 ZIP 옆 flylm_runs/실행시각/에 저장된다.

로컬에서는 JSON 구조 확인과 cold test만 수행했다. 실제 ZIP 전처리/Colab 학습은 아직 실행하지 않았다.
MaleCNS 최초 실행은 공식 데이터 약 1.11 GB 다운로드가 추가된다.


## 1. 런타임과 학습 예산

**8,192 neurons / batch 1 / TBPTT 8 tokens / 2 brain steps / FP32 / token checkpointing**.
train_updates는 화자2 target을 포함하는 chunk의 optimizer update 상한이다. 화자1 입력만 있는 chunk는 세지 않는다.
eval_every=200마다 고정 validation 표본을 평가하고 save_every=200마다 latest를 Drive에 저장한다.
T4에서 BF16을 전제하지 않는다. 없는 보조 라이브러리만 설치하며 기존 torch는 교체하지 않는다.
CUDA reduction 때문에 bitwise 재현은 보장되지 않는다.


In [1]:
import importlib.util
import subprocess
import sys

required = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy',
            'pyarrow': 'pyarrow', 'tqdm': 'tqdm'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('CUDA PyTorch가 제공되는 Colab GPU 런타임을 선택하세요.')

import gc
import hashlib
import json
import math
import platform
import random
import shutil
import time
import urllib.request
import zipfile
from datetime import datetime, timezone
from collections import deque
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import scipy
from scipy import sparse
from scipy.sparse.csgraph import connected_components, breadth_first_order
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

CFG = dict(seed=42, n_neurons=8192, superclasses=['cb_intrinsic'],
           n_inputs=128, n_outputs=128, embedding_dim=32, brain_steps=2,
           batch_size=1, tbptt=8, train_updates=1000, epochs=1, lr=0.003,
           checkpoint_tokens=True, nt_confidence=0.8, use_nt_signs=True,
           max_candidate_edges=8_000_000, max_model_edges=2_000_000,
           log_every=50, eval_every=200, save_every=200,
           eval_max_pairs=8, eval_response_tokens=512)
CACHE = Path('flylm_malecns_cache')
CACHE.mkdir(exist_ok=True)
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if not torch.cuda.is_available():
    raise RuntimeError('이 노트북은 원격 CUDA GPU 런타임용입니다. T4를 선택하고 다시 실행하세요.')
torch.cuda.manual_seed_all(CFG['seed'])
device = torch.device('cuda')
VERSIONS = dict(python=platform.python_version(), torch=torch.__version__,
                numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__, pyarrow=pa.__version__)
print(VERSIONS)
print(torch.cuda.get_device_name(0), 'GPU GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)
assert CFG['n_neurons'] == 8192
assert CFG['batch_size'] == 1, '입력–답변 경계를 보존하는 현재 로더는 batch 1 기준입니다.'

assert CFG['epochs'] >= 1 and CFG['tbptt'] >= 1
assert CFG['train_updates'] is None or CFG['train_updates'] >= 1
assert min(CFG[k] for k in ['log_every', 'eval_every', 'save_every', 'eval_max_pairs', 'eval_response_tokens']) >= 1


{'python': '3.13.15', 'torch': '2.11.0+cu128', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'pyarrow': '18.1.0'}
Tesla T4 GPU GiB: 14.56317138671875


## 1A. Drive

/content/drive/MyDrive/text_dataset/dataset/일상대화_데이터.zip을 사용한다.
파일명이 macOS NFD 한글로 저장된 경우에만 같은 디렉터리에서 Unicode 정규화한 동일 파일명을 찾아 실제 경로를 출력한다.
원본 ZIP은 수정하지 않으며 VM 로컬 복사본을 읽는다.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import unicodedata
DATA_CFG = dict(zip_path='/content/drive/MyDrive/text_dataset/dataset/일상대화_데이터.zip',
                text_field='form', validation_fraction=0.05)
requested_zip = Path(DATA_CFG['zip_path'])
ZIP_PATH = requested_zip if requested_zip.is_file() else None
if ZIP_PATH is None and requested_zip.parent.is_dir():
    matches = [p for p in requested_zip.parent.iterdir() if p.is_file()
               and unicodedata.normalize('NFC', p.name) == requested_zip.name]
    if len(matches) == 1:
        ZIP_PATH = matches[0]
if ZIP_PATH is None:
    raise FileNotFoundError(f'ZIP을 확인하세요: {requested_zip}')
print('학습 데이터 ZIP:', ZIP_PATH)
DATA_CACHE = Path('/content/flylm_dialogue_cache')
DATA_CACHE.mkdir(exist_ok=True)
local_zip = DATA_CACHE / 'dialogues.zip'
stamp_path = DATA_CACHE / 'zip_source.json'
zip_stat = ZIP_PATH.stat()
stamp = dict(path=str(ZIP_PATH), size=zip_stat.st_size, mtime_ns=zip_stat.st_mtime_ns)
old_stamp = json.loads(stamp_path.read_text()) if stamp_path.exists() else None
if old_stamp != stamp or not local_zip.exists() or local_zip.stat().st_size != zip_stat.st_size:
    if shutil.disk_usage(DATA_CACHE).free < zip_stat.st_size + 512 * 1024**2:
        raise RuntimeError('ZIP 로컬 복사본을 위한 디스크 공간이 부족합니다.')
    partial = local_zip.with_suffix('.partial')
    shutil.copyfile(ZIP_PATH, partial)
    if partial.stat().st_size != zip_stat.st_size:
        raise RuntimeError('ZIP 복사 크기가 일치하지 않습니다.')
    partial.replace(local_zip)
    stamp_path.write_text(json.dumps(stamp), encoding='utf-8')
digest = hashlib.sha256()
with local_zip.open('rb') as f:
    for block in iter(lambda: f.read(8 * 1024**2), b''):
        digest.update(block)
zip_sha256 = digest.hexdigest()
print('ZIP SHA-256:', zip_sha256)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
학습 데이터 ZIP: /content/drive/MyDrive/text_dataset/dataset/일상대화_데이터.zip
ZIP SHA-256: f5661f9a71481bbacd5aabdcbb16c9a2dee174556e7ba1d0e4cdc0264f653f20


## 1B. 두 화자의 입력–답변 쌍과 문서 단위 split

JSON 배열 순서를 유지한다. form의 연속 공백만 정리하고 original_form/인구통계/시간 metadata는 학습 텍스트에 넣지 않는다.
text_field를 original_form으로 바꾸면 그 필드로 다시 전처리한다. 지정 필드의 누락/잘못된 타입은 위치를 표시하고 중단한다.
speaker_id로 화자를 구분하며 동일 화자의 연속 발화를 하나로 합친다. 마지막 화자1에 답변이 없으면 해당 turn은 쓰지 않는다.

각 학습 쌍은 다음 구조다.

    BOS + "화자1: 입력 내용\n화자2: " + 답변 내용 + EOS
    └──────────── loss 제외 ───────────┘  └── loss 적용 ──┘

쌍마다 prompt_tokens 경계를 기록한다. uint16 토큰 캐시에 입력과 답변을 저장하되, prompt는 label -100으로 mask한다.
중복 대화/쌍을 제거한 후 대화 단위 95:5 분리를 seed와 hash로 고정한다.
같은 화자가 다른 문서에도 있을 수 있으므로 화자 독립 평가는 아니다.
최소 2개 유효 대화가 필요하며, 조건에 맞는 쌍이 없으면 가짜 답변을 만들지 않고 중단한다.


In [4]:
def dialogue_turns(document, location):
    utterances = document.get('utterance')
    if not isinstance(utterances, list):
        raise ValueError(f'{location}: utterance must be a list')
    speakers, turns = {}, []
    for index, utterance in enumerate(utterances):
        if not isinstance(utterance, dict):
            raise ValueError(f'{location}: utterance[{index}] must be an object')
        text = utterance.get(DATA_CFG['text_field'])
        if not isinstance(text, str):
            raise ValueError(f'{location}: utterance[{index}].{DATA_CFG["text_field"]} must be a string')
        text = ' '.join(text.split())
        if not text:
            continue
        speaker = utterance.get('speaker_id')
        if not isinstance(speaker, str) or not speaker.strip():
            return [], 'missing_speaker'
        if speaker not in speakers:
            speakers[speaker] = len(speakers) + 1
        role = speakers[speaker]
        if turns and turns[-1][0] == role:
            turns[-1][1].append(text)
        else:
            turns.append((role, [text]))
    if not turns:
        return [], 'empty'
    if len(speakers) != 2:
        return [], 'single_speaker' if len(speakers) == 1 else 'more_than_two_speakers'
    return [(role, ' '.join(parts)) for role, parts in turns], None

def encode_pair(question, answer):
    prefix = ('화자1: ' + question + '\n화자2: ').encode('utf-8')
    reply = answer.encode('utf-8')
    tokens = np.empty(1 + len(prefix) + len(reply) + 1, dtype='<u2')
    tokens[0], tokens[-1] = 256, 257
    tokens[1:1+len(prefix)] = np.frombuffer(prefix, dtype=np.uint8)
    tokens[1+len(prefix):-1] = np.frombuffer(reply, dtype=np.uint8)
    return tokens, 1 + len(prefix)

data_spec = dict(version=2, zip_sha256=zip_sha256, text_field=DATA_CFG['text_field'],
                 renderer='first_seen_two_speakers_adjacent_1to2_merged_turns_v1',
                 tokenizer='utf8_byte_bos256_eos257_uint16le', loss='response_and_eos_only',
                 seed=CFG['seed'], validation_fraction=DATA_CFG['validation_fraction'])
assert 0 < DATA_CFG['validation_fraction'] < 1
data_key = hashlib.sha256(json.dumps(data_spec, sort_keys=True).encode()).hexdigest()[:16]
token_path = DATA_CACHE / f'{data_key}.uint16'
index_path = DATA_CACHE / f'{data_key}.json'


def prepare_pairs():
    documents, pairs, seen_ids = [], [], {}
    seen_docs, seen_pairs, skipped = set(), set(), {}
    offset = 0
    partial = token_path.with_suffix('.partial')
    with zipfile.ZipFile(local_zip) as archive, partial.open('wb') as output:
        members = sorted((info for info in archive.infolist()
                          if not info.is_dir() and info.filename.lower().endswith('.json')
                          and not info.filename.startswith('__MACOSX/')
                          and not info.filename.rsplit('/', 1)[-1].startswith('._')),
                         key=lambda info: info.filename)
        if not members:
            raise ValueError('ZIP 안에 JSON 파일이 없습니다.')
        for info in tqdm(members, desc='JSON → speaker1/speaker2 pairs'):
            with archive.open(info) as member:
                payload = json.loads(member.read().decode('utf-8-sig'))
            if not isinstance(payload, dict) or not isinstance(payload.get('document'), list):
                raise ValueError(f'{info.filename}: document list가 없습니다.')
            for number, document in enumerate(payload['document']):
                if not isinstance(document, dict):
                    raise ValueError(f'{info.filename}: document[{number}] must be an object')
                doc_id = str(document.get('id') or f'{info.filename}#{number}')
                turns, reason = dialogue_turns(document, f'{info.filename}:{doc_id}')
                if reason:
                    skipped[reason] = skipped.get(reason, 0) + 1
                    continue
                doc_hash = hashlib.sha256(json.dumps(turns, ensure_ascii=False).encode()).hexdigest()
                if doc_id in seen_ids and seen_ids[doc_id] != doc_hash:
                    raise ValueError(f'Duplicate document ID with different content: {doc_id}')
                seen_ids[doc_id] = doc_hash
                if doc_hash in seen_docs:
                    skipped['duplicate_document'] = skipped.get('duplicate_document', 0) + 1
                    continue
                seen_docs.add(doc_hash)
                count = 0
                for i in range(len(turns)-1):
                    if turns[i][0] != 1 or turns[i+1][0] != 2:
                        continue
                    question, answer = turns[i][1], turns[i+1][1]
                    pair_hash = hashlib.sha256(json.dumps([question, answer], ensure_ascii=False).encode()).hexdigest()
                    if pair_hash in seen_pairs:
                        skipped['duplicate_pair'] = skipped.get('duplicate_pair', 0) + 1
                        continue
                    seen_pairs.add(pair_hash)
                    tokens, prompt_tokens = encode_pair(question, answer)
                    tokens.tofile(output)
                    pairs.append(dict(document=len(documents), sha256=pair_hash, offset=offset,
                                      length=len(tokens), prompt_tokens=prompt_tokens))
                    offset += len(tokens)
                    count += 1
                if count:
                    documents.append(dict(id=doc_id, member=info.filename, sha256=doc_hash, pairs=count))
                else:
                    skipped['no_unique_pair'] = skipped.get('no_unique_pair', 0) + 1
    if len(documents) < 2:
        raise ValueError(f'유효한 2인 대화가 {len(documents)}개입니다. train/validation에 최소 2개 필요합니다. 제외: {skipped}')
    order = sorted(range(len(documents)), key=lambda i: hashlib.sha256(
        f'{CFG["seed"]}:{documents[i]["sha256"]}'.encode()).hexdigest())
    n_val = min(len(documents)-1, max(1, round(len(documents)*DATA_CFG['validation_fraction'])))
    validation = set(order[:n_val])
    for i, document in enumerate(documents):
        document['split'] = 'validation' if i in validation else 'train'
    for pair in pairs:
        pair['split'] = documents[pair['document']]['split']
    manifest = dict(spec=data_spec, documents=documents, pairs=pairs, total_tokens=offset,
                    json_files=len(members), skipped=skipped)
    partial.replace(token_path)
    index_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    return manifest

dataset_manifest = json.loads(index_path.read_text(encoding='utf-8')) if index_path.exists() else None
if (dataset_manifest is None or dataset_manifest['spec'] != data_spec or not token_path.exists()
        or token_path.stat().st_size != dataset_manifest['total_tokens']*2):
    dataset_manifest = prepare_pairs()
documents, pairs = dataset_manifest['documents'], dataset_manifest['pairs']
tokens_mm = np.memmap(token_path, mode='r', dtype='<u2')
train_ids = [i for i, row in enumerate(pairs) if row['split'] == 'train']
val_ids = [i for i, row in enumerate(pairs) if row['split'] == 'validation']
assert train_ids and val_ids
assert {pairs[i]['document'] for i in train_ids}.isdisjoint({pairs[i]['document'] for i in val_ids})
assert {pairs[i]['sha256'] for i in train_ids}.isdisjoint({pairs[i]['sha256'] for i in val_ids})
assert sum(row['length'] for row in pairs) == len(tokens_mm)
for row in pairs:
    start, end = row['offset'], row['offset'] + row['length']
    assert int(tokens_mm[start]) == 256 and int(tokens_mm[end-1]) == 257
    assert 1 < row['prompt_tokens'] < row['length']-1
data_stats = dict(json_files=dataset_manifest['json_files'], usable_documents=len(documents),
                  train_documents=len({pairs[i]['document'] for i in train_ids}),
                  validation_documents=len({pairs[i]['document'] for i in val_ids}),
                  train_pairs=len(train_ids), validation_pairs=len(val_ids),
                  train_response_tokens=sum(pairs[i]['length']-pairs[i]['prompt_tokens'] for i in train_ids),
                  validation_response_tokens=sum(pairs[i]['length']-pairs[i]['prompt_tokens'] for i in val_ids),
                  excluded=dataset_manifest['skipped'])
print(json.dumps(data_stats, ensure_ascii=False, indent=2))
RUN_DIR = ZIP_PATH.parent / 'flylm_runs' / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR.mkdir(parents=True, exist_ok=False)
shutil.copyfile(index_path, RUN_DIR / 'dataset_manifest.json')
print('Drive 결과 저장 경로:', RUN_DIR)


JSON → speaker1/speaker2 pairs:   0%|          | 0/14229 [00:00<?, ?it/s]

{
  "json_files": 14229,
  "usable_documents": 11520,
  "train_documents": 10944,
  "validation_documents": 576,
  "train_pairs": 331601,
  "validation_pairs": 17687,
  "train_response_tokens": 73720635,
  "validation_response_tokens": 3910781,
  "excluded": {
    "single_speaker": 380,
    "duplicate_pair": 749,
    "more_than_two_speakers": 2329
  }
}
Drive 결과 저장 경로: /content/drive/MyDrive/text_dataset/dataset/flylm_runs/20260907_151606_577848


## 2. 공식 데이터와 캐시

[MaleCNS 공식 다운로드](https://male-cns.janelia.org/download/)의 annotation, neuron-level neurotransmitter,
segment-to-segment connectivity 세 파일만 사용한다. synapse 좌표나 원본 EM은 받지 않는다.
다운로드 중단 파일은 `.part`로 남고 재실행 시 처음부터 다시 받는다. 완성 파일의 길이를 확인하고 SHA-256을 기록한다.
SHA-256은 이 실행의 출처 추적용이며 제공자가 발표한 checksum과 대조한 것은 아니다.

공식 graph에는 작은 segment도 포함된다. **annotation `status == Traced` 및 `superclass == cb_intrinsic`**로
후보를 제한해 glia/미추적 segment를 제외한다. 모든 MaleCNS를 대표하는 표본이 아니라 central-brain 내부 회로 실험이다.


In [5]:
BASE_URL = 'https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/'
SOURCES = {
    'annotations': ('body-annotations-male-cns-v1.0-minconf-0.5.feather', 14483314),
    'nt': ('body-neurotransmitters-male-cns-v1.0.feather', 43282834),
    'edges': ('connectome-weights-male-cns-v1.0-minconf-0.5.feather', 1051241946),
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()

def download_file(name, expected_size):
    target = CACHE / name
    if not target.exists() or target.stat().st_size != expected_size:
        if shutil.disk_usage(CACHE).free < expected_size + 512 * 1024**2:
            raise RuntimeError('데이터 다운로드에 필요한 디스크 공간이 부족합니다.')
        partial = target.with_suffix(target.suffix + '.part')
        with urllib.request.urlopen(BASE_URL + name, timeout=60) as response:
            with partial.open('wb') as f, tqdm(total=expected_size, unit='B', unit_scale=True, desc=name) as bar:
                while True:
                    chunk = response.read(8 * 1024**2)
                    if not chunk:
                        break
                    f.write(chunk)
                    bar.update(len(chunk))
        if partial.stat().st_size != expected_size:
            raise RuntimeError(f'파일 크기 불일치: {name}. v1.0 파일/다운로드를 확인하세요.')
        partial.replace(target)
    return target

files = {key: download_file(*spec) for key, spec in SOURCES.items()}
source_manifest = {key: dict(url=BASE_URL + p.name, bytes=p.stat().st_size, sha256=sha256_file(p))
                   for key, p in files.items()}
print({key: entry['bytes'] for key, entry in source_manifest.items()})


body-annotations-male-cns-v1.0-minconf-0.5.feather:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

body-neurotransmitters-male-cns-v1.0.feather:   0%|          | 0.00/43.3M [00:00<?, ?B/s]

connectome-weights-male-cns-v1.0-minconf-0.5.feather:   0%|          | 0.00/1.05G [00:00<?, ?B/s]

{'annotations': 14483314, 'nt': 43282834, 'edges': 1051241946}


## 3. Batch 단위 원본 그래프 읽기 → 연결된 8,192개 선택

전체 connection table을 pandas로 한꺼번에 올리지 않는다. Arrow record batch를 읽으며 후보 사이의 edge만 보관한다.
source/destination은 `int64` 원본 body ID에서 연속 정수로 변환하고 중복 pair의 count를 합친다.
강연결요소(SCC) 중 가장 큰 영역에서 weighted out-degree가 가장 큰 뉴런을 seed로 선택한 뒤 유향 BFS 순서의 첫 8,192개를 뽑는다.
선택된 노드 사이의 **모든 원본 edge**를 보존한다. self-edge가 원본에 있으면 유지한다.

부분 그래프에서는 바깥으로 향하는 연결이 사라진다. BFS prefix는 seed에서 모두 도달 가능하지만
그 자체가 강연결일 필요는 없다. SCC 크기와 연결성은 실제로 출력한다.
edge 한도를 넘으면 명시적으로 중단하며 edge를 몰래 pruning하거나 가짜 연결을 추가하지 않는다.


In [6]:
def require_columns(names, required, label):
    absent = set(required) - set(names)
    if absent:
        raise ValueError(f'{label}: missing {sorted(absent)}; found {list(names)}')

def map_ids(sorted_ids, values):
    positions = np.searchsorted(sorted_ids, values)
    safe = positions.clip(0, len(sorted_ids) - 1)
    valid = (positions < len(sorted_ids)) & (sorted_ids[safe] == values)
    return positions, valid

def load_candidate_graph(edge_file, ids):
    pieces = []
    count = 0
    with pa.memory_map(str(edge_file), 'r') as source:
        reader = pa.ipc.open_file(source)
        require_columns(reader.schema.names, ['body_pre', 'body_post', 'weight'], 'connectome')
        for i in tqdm(range(reader.num_record_batches), desc='Filter connection batches'):
            batch = reader.get_batch(i)
            pre = batch.column('body_pre').to_numpy(zero_copy_only=False)
            post = batch.column('body_post').to_numpy(zero_copy_only=False)
            weight = batch.column('weight').to_numpy(zero_copy_only=False)
            s, valid_s = map_ids(ids, pre)
            d, valid_d = map_ids(ids, post)
            keep = valid_s & valid_d
            if not keep.any():
                continue
            w = weight[keep]
            if not np.isfinite(w).all() or (w <= 0).any():
                raise ValueError('Nonpositive/null synapse count in selected candidates')
            count += int(keep.sum())
            if count > CFG['max_candidate_edges']:
                raise RuntimeError('Candidate edge RAM guard: superclasses를 더 좁히거나 host RAM 확인 후 한도를 조정하세요.')
            pieces.append((s[keep].astype(np.int32), d[keep].astype(np.int32), w.astype(np.int64)))
    if not pieces:
        raise RuntimeError('선택 조건에 맞는 연결이 없습니다.')
    s = np.concatenate([p[0] for p in pieces])
    d = np.concatenate([p[1] for p in pieces])
    w = np.concatenate([p[2] for p in pieces])
    del pieces
    graph = sparse.csr_matrix((w, (s, d)), shape=(len(ids), len(ids)))
    graph.sum_duplicates()
    graph.sort_indices()
    return graph

def build_subgraph():
    annotation = feather.read_table(files['annotations']).to_pandas()
    require_columns(annotation.columns, ['bodyId', 'status', 'superclass'], 'annotations')
    eligible = annotation.loc[(annotation['status'] == 'Traced') &
                              annotation['superclass'].isin(CFG['superclasses'])].copy()
    if eligible['bodyId'].isna().any() or eligible['bodyId'].duplicated().any():
        raise ValueError('Invalid/duplicate annotation body IDs')
    ids = np.sort(eligible['bodyId'].to_numpy(dtype=np.int64))
    if len(ids) < CFG['n_neurons']:
        raise RuntimeError(f'후보 뉴런 부족: {len(ids)}')
    graph = load_candidate_graph(files['edges'], ids)
    _, labels = connected_components(graph, directed=True, connection='strong')
    component_sizes = np.bincount(labels)
    component = np.flatnonzero(labels == component_sizes.argmax())
    if len(component) < CFG['n_neurons']:
        raise RuntimeError(f'최대 SCC가 {len(component)}개입니다. 추출 범위를 넓혀야 합니다.')
    core = graph[component][:, component].tocsr()
    seed = int(np.asarray(core.sum(axis=1)).ravel().argmax())
    order = breadth_first_order(core, seed, directed=True, return_predecessors=False)
    selected = component[order[:CFG['n_neurons']]]
    sub = graph[selected][:, selected].tocsr()
    if sub.nnz > CFG['max_model_edges']:
        raise RuntimeError(f'{sub.nnz:,} edges: GPU guard 초과. 원본 edge를 보존하므로 자동 삭제하지 않습니다.')
    assert len(selected) == CFG['n_neurons']
    assert len(breadth_first_order(sub, 0, directed=True, return_predecessors=False)) == len(selected)
    coo = sub.tocoo()
    result = dict(body_ids=ids[selected], src=coo.row.astype(np.int64), dst=coo.col.astype(np.int64),
                  counts=coo.data.astype(np.int64))
    stats = dict(candidate_neurons=int(len(ids)), candidate_edges=int(graph.nnz),
                 candidate_largest_scc=int(len(component)), seed_body_id=int(ids[selected[0]]))
    return result, stats

extract_spec = dict(version=1, n_neurons=CFG['n_neurons'], superclasses=CFG['superclasses'],
                    status='Traced', selection='largest_SCC_weighted_outdegree_seed_directed_BFS',
                    source_sha256={k: v['sha256'] for k, v in source_manifest.items()},
                    scipy=VERSIONS['scipy'])
cache_key = hashlib.sha256(json.dumps(extract_spec, sort_keys=True).encode()).hexdigest()[:16]
graph_path = CACHE / f'graph_{cache_key}.npz'
meta_path = CACHE / f'graph_{cache_key}.json'
if graph_path.exists() and meta_path.exists():
    with np.load(graph_path, allow_pickle=False) as saved:
        graph_data = {k: saved[k] for k in saved.files}
    extraction_stats = json.loads(meta_path.read_text())['stats']
else:
    graph_data, extraction_stats = build_subgraph()
    np.savez_compressed(graph_path, **graph_data)
    meta_path.write_text(json.dumps(dict(spec=extract_spec, stats=extraction_stats), indent=2))
assert len(graph_data['body_ids']) == CFG['n_neurons']
assert len(graph_data['src']) <= CFG['max_model_edges']
print(extraction_stats)
gc.collect()


Filter connection batches:   0%|          | 0/2318 [00:00<?, ?it/s]

{'candidate_neurons': 32160, 'candidate_edges': 6814804, 'candidate_largest_scc': 32159, 'seed_body_id': 10977}


14267

## 4. 신경전달물질 해석과 입력/출력 포트

실제 컬럼은 `body`, `ground_truth`, `predicted_nt`, `predicted_nt_confidence`, `consensus_nt`다.
`ground_truth`가 알려진 전달물질이면 우선 사용하고, 없으면 neuron prediction confidence ≥ 0.8일 때만 그 label을 쓴다.
`consensus_nt`는 출처와 함께 보존하지만 neuron prediction confidence를 consensus의 confidence로 잘못 사용하지 않는다.

모델링 가정: **acetylcholine → 양수, GABA → 음수**를 고정한다. receptor별 예외를 재현하는 생물학적 시뮬레이션은 아니다.
glutamate·histamine·monoamine·불확실 label은 부호를 강제하지 않는다. 특히 glutamate의 효과는 수용체/영역에 의존한다.
예측 confidence는 확률적으로 보정된 정확도를 보장하지 않는다.

입력은 seed와 out-degree가 큰 127개, 출력은 입력에서 도달 가능한 별도 128개를 고른다.
짧은 경로를 먼저 선택하고 동률이면 incoming synapse count를 우선한다. 이는 공학적 adapter 위치이며 sensory/motor라는 주장이 아니다.


In [7]:
KNOWN_NT = {'acetylcholine', 'gaba', 'glutamate', 'histamine', 'dopamine', 'serotonin', 'octopamine'}

def attach_nt(body_ids):
    selected_parts = []
    wanted = np.sort(body_ids)
    with pa.memory_map(str(files['nt']), 'r') as source:
        reader = pa.ipc.open_file(source)
        require_columns(reader.schema.names, ['body', 'ground_truth', 'predicted_nt',
                                              'predicted_nt_confidence', 'consensus_nt'], 'neurotransmitters')
        for i in range(reader.num_record_batches):
            batch = reader.get_batch(i)
            _, keep = map_ids(wanted, batch.column('body').to_numpy(zero_copy_only=False))
            if keep.any():
                selected_parts.append(batch.filter(pa.array(keep)).to_pandas())
    if not selected_parts:
        raise RuntimeError('선택된 body와 전달물질 테이블의 join이 비어 있습니다.')
    table = pd.concat(selected_parts, ignore_index=True)
    if table['body'].duplicated().any():
        raise ValueError('Duplicate NT body IDs: explicit aggregation is required')
    table = table.set_index('body').reindex(body_ids)
    truth = table['ground_truth'].fillna('').str.strip().str.lower()
    predicted = table['predicted_nt'].fillna('').str.strip().str.lower()
    confidence = pd.to_numeric(table['predicted_nt_confidence'], errors='coerce')
    trusted = truth.isin(KNOWN_NT)
    confident = predicted.isin(KNOWN_NT) & confidence.ge(CFG['nt_confidence']) & confidence.le(1)
    effective = np.where(trusted, truth, np.where(confident, predicted, 'unknown'))
    origin = np.where(trusted, 'ground_truth', np.where(confident, 'predicted_nt', 'unknown'))
    table['effective_nt'] = effective
    table['effective_source'] = origin
    table.index.name = 'body'
    return table

def distances(graph, starts):
    distance = np.full(graph.shape[0], -1, dtype=np.int32)
    queue = deque(int(x) for x in starts)
    distance[np.asarray(starts, dtype=np.int64)] = 0
    while queue:
        i = queue.popleft()
        for j in graph.indices[graph.indptr[i]:graph.indptr[i+1]]:
            if distance[j] == -1:
                distance[j] = distance[i] + 1
                queue.append(int(j))
    return distance

n = CFG['n_neurons']
src, dst, counts = (graph_data[k] for k in ['src', 'dst', 'counts'])
assert len(np.unique(graph_data['body_ids'])) == n
assert src.shape == dst.shape == counts.shape and len(src) > 0
assert src.min() >= 0 and dst.min() >= 0 and src.max() < n and dst.max() < n
assert (counts > 0).all() and np.isfinite(counts).all()
assert len(np.unique(src * n + dst)) == len(src)
adj = sparse.csr_matrix((counts, (src, dst)), shape=(n, n))  # row=source, col=destination
out_strength = np.asarray(adj.sum(axis=1)).ravel()
in_strength = np.asarray(adj.sum(axis=0)).ravel()
rank = np.argsort(-out_strength, kind='stable')
input_nodes = np.array([0] + [int(i) for i in rank if i != 0][:CFG['n_inputs']-1], dtype=np.int64)
dist = distances(adj, input_nodes)
choices = np.flatnonzero(dist > 0)
if len(choices) < CFG['n_outputs']:
    raise RuntimeError('Disjoint reachable output ports 부족')
ranked = np.lexsort((choices, -in_strength[choices], dist[choices]))
output_nodes = choices[ranked[:CFG['n_outputs']]].astype(np.int64)
can_reach_output = distances(adj.T.tocsr(), output_nodes) >= 0
_, sub_labels = connected_components(adj, directed=True, connection='strong')
nt_table = attach_nt(graph_data['body_ids'])
node_sign = nt_table['effective_nt'].map({'acetylcholine': 1.0, 'gaba': -1.0}).fillna(0).to_numpy(np.float32)
if not CFG['use_nt_signs']:
    node_sign[:] = 0  # 명시적인 ablation, 기본값은 True
edge_sign = node_sign[src]
if CFG['use_nt_signs'] and not np.any(edge_sign):
    raise RuntimeError('적용 가능한 NT sign이 0개입니다. join/label/confidence를 확인하세요.')
graph_stats = dict(neurons=n, edges=int(len(src)), self_edges=int((src == dst).sum()),
                   largest_scc=int(np.bincount(sub_labels).max()),
                   reachable_from_inputs=int((dist >= 0).sum()),
                   can_reach_outputs=int(can_reach_output.sum()),
                   input_to_output_max_hops=int(dist[output_nodes].max()),
                   positive_edges=int((edge_sign > 0).sum()), negative_edges=int((edge_sign < 0).sum()),
                   unconstrained_edges=int((edge_sign == 0).sum()))
print(graph_stats)
print(nt_table.groupby(['effective_source', 'effective_nt']).size().to_string())
assert graph_stats['reachable_from_inputs'] == n
assert len(set(input_nodes) & set(output_nodes)) == 0
if graph_stats['can_reach_outputs'] < n:
    print('주의: 일부 뉴런은 선택된 출력에 도달하지 못해 next-token loss로 학습되지 않을 수 있습니다.')
nt_table.to_csv(CACHE / f'neurotransmitters_{cache_key}.csv')


{'neurons': 8192, 'edges': 1327807, 'self_edges': 7, 'largest_scc': 8191, 'reachable_from_inputs': 8192, 'can_reach_outputs': 8191, 'input_to_output_max_hops': 1, 'positive_edges': 353628, 'negative_edges': 57235, 'unconstrained_edges': 916944}
effective_source  effective_nt 
ground_truth      acetylcholine     605
                  dopamine          312
                  gaba              127
                  glutamate         201
                  octopamine         21
                  serotonin          11
predicted_nt      acetylcholine    2201
                  dopamine          143
                  gaba              182
                  glutamate         543
                  serotonin           8
unknown           unknown          3838
주의: 일부 뉴런은 선택된 출력에 도달하지 못해 next-token loss로 학습되지 않을 수 있습니다.


## 5. 커넥톰 recurrent LM

`prior_e = 0.9 × log(1 + synapse_count_e) / sum_incoming(log(1 + count))`로 초기화한다.
이 정규화는 초기 recurrent 입력의 크기를 제한하기 위한 공학적 선택이며 실제 synaptic efficacy가 아니다.
부호가 있는 edge는 `prior × sign × softplus(theta)/log(2)`, 나머지는 `prior × theta`로 학습한다.
초기 theta는 부호 제약 edge에서 0, 나머지에서 ±1이다. 학습 후에도 알려진 부호와 topology는 유지된다.

`h ← 0.5 h + 0.5 tanh(weighted_messages + input + bias)`를 토큰마다 두 번 수행한다.
누설 상태 유지는 자기 시간 기억이다. 입력/출력 adapter 외의 뉴런 간 연결은 원본 edge 목록뿐이다.
FP32를 유지하고 token 단위 activation checkpoint로 B×E 중간 텐서의 보관을 줄인다.
이 모델에서 N×N dense adjacency는 만들지 않는다.


In [8]:
BOS, EOS, VOCAB = 256, 257, 258

def encode(text):
    return [BOS] + list(text.encode('utf-8')) + [EOS]

def decode(ids):
    return bytes(int(i) for i in ids if 0 <= int(i) < 256).decode('utf-8', errors='replace')

class ConnectomeLM(nn.Module):
    def __init__(self, cfg, src, dst, counts, edge_sign, inputs, outputs):
        super().__init__()
        self.n = cfg['n_neurons']
        self.k = cfg['brain_steps']
        self.checkpoint_tokens = cfg['checkpoint_tokens']
        for name, value in [('src', src), ('dst', dst), ('inputs', inputs), ('outputs', outputs)]:
            self.register_buffer(name, torch.as_tensor(np.array(value, copy=True), dtype=torch.long))
        sign = torch.as_tensor(np.array(edge_sign, copy=True), dtype=torch.float32)
        c = torch.as_tensor(np.array(counts, copy=True), dtype=torch.float32).log1p()
        incoming = torch.zeros(self.n).index_add(0, self.dst, c)
        self.register_buffer('prior', 0.9 * c / incoming[self.dst].clamp_min(1e-8))
        self.register_buffer('edge_sign', sign)
        raw = torch.where(sign != 0, torch.zeros_like(sign),
                          torch.where(torch.rand_like(sign) < 0.5, -torch.ones_like(sign), torch.ones_like(sign)))
        self.theta = nn.Parameter(raw)
        self.bias = nn.Parameter(torch.zeros(self.n))
        self.embedding = nn.Embedding(VOCAB, cfg['embedding_dim'])
        self.input_adapter = nn.Linear(cfg['embedding_dim'], len(inputs))
        self.output_adapter = nn.Linear(len(outputs), cfg['embedding_dim'])
        self.output_bias = nn.Parameter(torch.zeros(VOCAB))

    def effective_weights(self):
        constrained = self.edge_sign * F.softplus(self.theta) / math.log(2)
        return self.prior * torch.where(self.edge_sign != 0, constrained, self.theta)

    def aggregate(self, h, weights):
        messages = h.index_select(1, self.src) * weights
        return torch.zeros_like(h).index_add(1, self.dst, messages)

    def advance(self, token, h):
        value = self.input_adapter(self.embedding(token))
        drive = torch.zeros_like(h).index_add(1, self.inputs, value)
        weights = self.effective_weights()
        for _ in range(self.k):
            h = 0.5 * h + 0.5 * torch.tanh(self.aggregate(h, weights) + drive + self.bias)
        return h

    def forward(self, tokens, state=None):
        if tokens.ndim != 2 or tokens.shape[1] == 0:
            raise ValueError('tokens must have shape [batch, nonempty time]')
        h = self.embedding.weight.new_zeros(tokens.shape[0], self.n) if state is None else state
        if h.shape != (tokens.shape[0], self.n):
            raise ValueError('state shape mismatch')
        logits = []
        for t in range(tokens.shape[1]):
            if self.training and self.checkpoint_tokens and torch.is_grad_enabled():
                h = checkpoint(self.advance, tokens[:, t], h, use_reentrant=False, preserve_rng_state=False)
            else:
                h = self.advance(tokens[:, t], h)
            z = self.output_adapter(h.index_select(1, self.outputs))
            logits.append(F.linear(z, self.embedding.weight, self.output_bias))
        return torch.stack(logits, dim=1), h

def make_model():
    torch.manual_seed(CFG['seed'])
    torch.cuda.manual_seed_all(CFG['seed'])
    return ConnectomeLM(CFG, src, dst, counts, edge_sign, input_nodes, output_nodes).to(device)

model = make_model()
parameters = sum(p.numel() for p in model.parameters())
# Adam parameter+gradient+two moments: 16 bytes/parameter. Buffers/activations/CUDA workspace는 별도다.
memory_lower = dict(parameters=parameters, adam_parameter_state_MiB=16 * parameters / 2**20,
                    one_BxE_message_MiB=4 * CFG['batch_size'] * len(src) / 2**20,
                    one_state_MiB=4 * CFG['batch_size'] * n / 2**20)
print(memory_lower)
print('위 수치는 메모리 하한 항목이며 GPU 전체 peak 추정값이 아닙니다.')
assert decode(encode('한국어 모델')) == '한국어 모델'


{'parameters': 1352865, 'adam_parameter_state_MiB': 20.643081665039062, 'one_BxE_message_MiB': 5.065181732177734, 'one_state_MiB': 0.03125}
위 수치는 메모리 하한 항목이며 GPU 전체 peak 추정값이 아닙니다.


## 6. 원격 실행용 검증: 작은 수치 기준 + 실제 8,192 모델

16개 **검사 전용 fixture**에서 dense 기준 값/gradient 및 checkpoint gradient를 비교한다. 실제 학습 그래프와 무관하다.
8,192개 모델에서는 짧은 입력으로 finite gradients, edge 갱신, 부호/연결 보존, 미래 토큰 인과성,
chunk와 통째 실행의 일치, state의 효과를 검사한다. 검사가 끝나면 학습 모델을 새로 초기화한다.


In [9]:
def numerical_checks():
    fixture = dict(CFG, n_neurons=16, embedding_dim=8, checkpoint_tokens=False)
    pairs = sorted({(i, (i + d) % 16) for i in range(16) for d in (1, 4)})
    s, d = np.asarray(pairs).T
    signs = np.resize(np.array([1, -1, 0], np.float32), len(s))
    m = ConnectomeLM(fixture, s, d, np.ones(len(s)), signs, np.arange(4), np.arange(4, 8)).to(device)
    h = torch.randn(2, 16, device=device, requires_grad=True)
    w = m.effective_weights()
    sparse_result = m.aggregate(h, w)
    dense_w = torch.zeros(16, 16, device=device).index_put((m.dst, m.src), w)
    dense_result = h @ dense_w.T
    torch.testing.assert_close(sparse_result, dense_result, rtol=1e-4, atol=1e-6)
    ga = torch.autograd.grad(sparse_result.square().sum(), (h, m.theta), retain_graph=True)
    gb = torch.autograd.grad(dense_result.square().sum(), (h, m.theta))
    for a, b in zip(ga, gb):
        torch.testing.assert_close(a, b, rtol=1e-4, atol=1e-6)
    tokens = torch.randint(0, 256, (1, 4), device=device)
    targets = torch.randint(0, 256, (1, 4), device=device)
    gradients = []
    for enabled in (False, True):
        m.zero_grad(set_to_none=True)
        m.checkpoint_tokens = enabled
        logits, _ = m(tokens)
        F.cross_entropy(logits.reshape(-1, VOCAB), targets.reshape(-1)).backward()
        gradients.append([p.grad.detach().clone() for p in m.parameters()])
    for a, b in zip(*gradients):
        torch.testing.assert_close(a, b, rtol=1e-4, atol=1e-6)
    print('PASS: fixture sparse/dense values + gradients; checkpoint gradients')

def real_graph_checks(m):
    tokens = torch.tensor([encode('가')], device=device)
    m.eval()
    with torch.no_grad():
        whole, end = m(tokens)
        first, state = m(tokens[:, :2])
        second, streamed_end = m(tokens[:, 2:], state)
        torch.testing.assert_close(whole, torch.cat([first, second], dim=1), rtol=1e-4, atol=1e-5)
        torch.testing.assert_close(end, streamed_end, rtol=1e-4, atol=1e-5)
        changed = tokens.clone()
        changed[:, 2:] = (changed[:, 2:] + 1) % VOCAB
        altered, _ = m(changed)
        torch.testing.assert_close(whole[:, :2], altered[:, :2], rtol=1e-4, atol=1e-5)
        reset, _ = m(tokens[:, 2:])
        state_effect = (reset - second).abs().max().item()
        assert state_effect > 1e-7, '출력에서 상태 효과가 관측되지 않습니다.'
    m.train()
    optimizer = torch.optim.Adam(m.parameters(), lr=CFG['lr'])
    before = m.theta.detach().clone()
    graph_before = (m.src.clone(), m.dst.clone())
    logits, _ = m(tokens[:, :-1])
    loss = F.cross_entropy(logits.reshape(-1, VOCAB), tokens[:, 1:].reshape(-1))
    loss.backward()
    for name, p in m.named_parameters():
        assert p.grad is not None and torch.isfinite(p.grad).all(), name
    assert m.theta.grad.abs().sum().item() > 0
    updated_fraction = (m.theta.grad.abs() > 0).float().mean().item()
    nn.utils.clip_grad_norm_(m.parameters(), 1.0, error_if_nonfinite=True)
    optimizer.step()
    assert not torch.equal(before, m.theta)
    assert torch.equal(graph_before[0], m.src) and torch.equal(graph_before[1], m.dst)
    w = m.effective_weights().detach()
    assert (w[m.edge_sign > 0] >= 0).all() and (w[m.edge_sign < 0] <= 0).all()
    assert not m.src.requires_grad and not m.dst.requires_grad
    print(dict(check='PASS: real graph forward/backward, topology, signs, causality, streaming',
               state_effect=state_effect, nonzero_edge_gradient_fraction=updated_fraction))

numerical_checks()
real_graph_checks(model)
del model
gc.collect()
torch.cuda.empty_cache()


PASS: fixture sparse/dense values + gradients; checkpoint gradients
{'check': 'PASS: real graph forward/backward, topology, signs, causality, streaming', 'state_effect': 0.03722113370895386, 'nonzero_edge_gradient_fraction': 0.9999955296516418}


## 7. Response-only TBPTT: 화자1은 조건, 화자2만 정답

모델은 teacher forcing으로 입력과 정답 답변의 이전 token을 순서대로 읽는다.
loss는 **화자2 답변 byte와 EOS**에만 적용한다. 화자1 내용, 화자 표기, BOS의 예측은 학습 target에서 제외한다.
각 입력–답변 쌍마다 state=None으로 시작한다. 쌍 사이에서는 hidden state를 넘기지 않는다.

입력만 있는 chunk는 no_grad로 상태를 계산하고, 답변 target이 있는 chunk에서만 backward/optimizer step을 수행한다.
chunk 사이에는 state.detach()를 적용한다. 따라서 긴 입력의 앞부분까지 gradient가 역전파되지는 않는다.
입력의 recurrent state는 답변에 전달되지만 TBPTT=8은 매우 짧은 gradient 문맥이다.
메모리와 속도를 확인한 뒤 tbptt를 늘리는 것이 긴 질문의 조건부 학습에 중요하다.

validation은 고정된 최대 8개 입력–답변 쌍에서 입력 전체를 읽고 답변 앞 최대 512 tokens만 평가한다.
input token을 분모에 넣지 않으며 response loss/accuracy/perplexity를 출력한다.
UTF-8 byte 및 EOS 기준 perplexity라 BPE 모델 값과 직접 비교하지 않는다.


In [10]:
IGNORE_INDEX = -100
assert (BOS, EOS) == (256, 257)

def response_mask(start, length, prompt_tokens):
    # x[k]가 예측하는 y[k]의 원래 sequence 위치는 start+k+1이다.
    return np.arange(start+1, start+length+1) >= prompt_tokens

def pair_chunks(pair_index, max_response_tokens=None):
    row = pairs[pair_index]
    stop = row['length']
    if max_response_tokens is not None:
        stop = min(stop, row['prompt_tokens'] + max_response_tokens)
    for start in range(0, stop-1, CFG['tbptt']):
        length = min(CFG['tbptt'], stop-1-start)
        offset = row['offset'] + start
        window = torch.from_numpy(np.array(tokens_mm[offset:offset+length+1], dtype=np.int64)).unsqueeze(0).to(device)
        labels = window[:, 1:].clone()
        mask = torch.from_numpy(response_mask(start, length, row['prompt_tokens'])).to(device).unsqueeze(0)
        labels.masked_fill_(~mask, IGNORE_INDEX)
        yield window[:, :-1], labels, int(mask.sum())

def check_response_format():
    fixture = dict(utterance=[
        dict(speaker_id='A', form='질문 하나'), dict(speaker_id='A', form='질문 둘'),
        dict(speaker_id='B', form='답변 하나'), dict(speaker_id='A', form='다음 질문'),
        dict(speaker_id='B', form='다음 답변')])
    # text_field 설정과 무관하게 동일 구조를 검사한다.
    for utterance in fixture['utterance']:
        utterance[DATA_CFG['text_field']] = utterance['form']
    turns, reason = dialogue_turns(fixture, 'fixture')
    assert reason is None and turns == [(1, '질문 하나 질문 둘'), (2, '답변 하나'), (1, '다음 질문'), (2, '다음 답변')]
    assert dialogue_turns(dict(utterance=fixture['utterance'][:2]), 'monologue')[1] == 'single_speaker'
    encoded, boundary = encode_pair('질문', '답변')
    actual_targets = []
    for start in range(0, len(encoded)-1, CFG['tbptt']):
        length = min(CFG['tbptt'], len(encoded)-1-start)
        mask = response_mask(start, length, boundary)
        actual_targets.extend(encoded[start+1:start+length+1][mask].tolist())
    assert actual_targets == list('답변'.encode('utf-8')) + [EOS]
    assert not response_mask(0, boundary-1, boundary).any()
    assert response_mask(boundary-1, 1, boundary)[0]  # 첫 답변 byte가 loss에 포함됨
    print('PASS: speaker mapping, turn merge, monologue exclusion, response-only mask + EOS')

check_response_format()
eval_ids = sorted(val_ids, key=lambda i: pairs[i]['sha256'])[:CFG['eval_max_pairs']]

@torch.no_grad()
def evaluate(m):
    was_training = m.training
    m.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for pair_index in eval_ids:
        state = None
        for x, labels, supervised in pair_chunks(pair_index, CFG['eval_response_tokens']):
            logits, state = m(x, state)
            if supervised:
                loss_sum += F.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1),
                                            ignore_index=IGNORE_INDEX, reduction='sum').item()
                correct += int(((logits.argmax(-1) == labels) & (labels != IGNORE_INDEX)).sum())
                total += supervised
    m.train(was_training)
    mean_loss = loss_sum/total
    if not math.isfinite(mean_loss):
        raise RuntimeError('Validation loss가 유한하지 않습니다.')
    return dict(response_loss=mean_loss, response_perplexity=math.exp(mean_loss) if mean_loss < 700 else None,
                response_accuracy=correct/total, response_tokens=total, pairs=len(eval_ids),
                full_validation=total == data_stats['validation_response_tokens'])

def to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: to_cpu(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(to_cpu(item) for item in value)
    return value

def save_checkpoint(name, m, optimizer, updates, validation):
    payload = dict(config=CFG, model_state=to_cpu(m.state_dict()), optimizer_state=to_cpu(optimizer.state_dict()),
                   graph={k: torch.from_numpy(v.copy()) for k, v in graph_data.items()},
                   edge_sign=torch.from_numpy(edge_sign.copy()),
                   input_nodes=torch.from_numpy(input_nodes.copy()), output_nodes=torch.from_numpy(output_nodes.copy()),
                   extract_spec=extract_spec, sources=source_manifest, data_spec=data_spec, data_config=DATA_CFG,
                   updates=updates, validation=validation)
    local = CACHE / name
    torch.save(payload, local)
    temporary = RUN_DIR / (name + '.partial')
    shutil.copyfile(local, temporary)
    temporary.replace(RUN_DIR / name)
    del payload

def save_history(rows):
    (RUN_DIR / 'history.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')


PASS: speaker mapping, turn merge, monologue exclusion, response-only mask + EOS


In [11]:
model = make_model()  # 이 셀 재실행은 새 학습이다.
optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'])
initial = dict(evaluate(model), evaluated_at_update=0)
best_loss, last_validation = initial['response_loss'], initial
history = [dict(update=0, validation=initial)]
save_checkpoint('best.pt', model, optimizer, 0, initial)
save_history(history)
updates, seen_response_tokens, processed_tokens, completed_epochs = 0, 0, 0, 0
pairs_started, pairs_completed = 0, 0
training_seconds, window_loss, window_tokens = 0.0, 0.0, 0

def supervised_updates(row):
    all_chunks = math.ceil((row['length']-1)/CFG['tbptt'])
    prompt_only_chunks = (row['prompt_tokens']-1)//CFG['tbptt']
    return all_chunks - prompt_only_chunks

updates_per_epoch = sum(supervised_updates(pairs[i]) for i in train_ids)
planned_updates = updates_per_epoch * CFG['epochs']
if CFG['train_updates'] is not None:
    planned_updates = min(planned_updates, CFG['train_updates'])
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
wall_start = time.perf_counter()
with tqdm(total=planned_updates, desc='Response train updates') as progress:
    for epoch in range(CFG['epochs']):
        order = train_ids.copy()
        random.Random(CFG['seed'] + epoch).shuffle(order)
        finished_in_epoch = 0
        for pair_index in order:
            if updates >= planned_updates:
                break
            state, pair_response_seen = None, 0
            pairs_started += 1
            for x, labels, supervised in pair_chunks(pair_index):
                if updates >= planned_updates:
                    break
                model.train()
                torch.cuda.synchronize()
                tick = time.perf_counter()
                if supervised == 0:
                    # 입력은 recurrent state에 반영하되 입력 자체를 정답으로 학습하지 않는다.
                    with torch.no_grad():
                        _, state = model(x, state)
                    state = state.detach()
                    torch.cuda.synchronize()
                    training_seconds += time.perf_counter() - tick
                    processed_tokens += x.numel()
                    continue
                optimizer.zero_grad(set_to_none=True)
                logits, next_state = model(x, state)
                loss = F.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1), ignore_index=IGNORE_INDEX)
                if not torch.isfinite(loss):
                    raise RuntimeError(f'Nonfinite response loss before update {updates+1}')
                loss.backward()
                grad_norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
                optimizer.step()
                state = next_state.detach()
                torch.cuda.synchronize()
                training_seconds += time.perf_counter() - tick
                updates += 1
                processed_tokens += x.numel()
                seen_response_tokens += supervised
                pair_response_seen += supervised
                window_loss += loss.item() * supervised
                window_tokens += supervised
                progress.update(1)
                if updates == 1 or updates % CFG['log_every'] == 0:
                    row = dict(update=updates, epoch=epoch+1, response_tokens=seen_response_tokens,
                               train_response_loss=window_loss/window_tokens, grad_norm=float(grad_norm))
                    history.append(row)
                    tqdm.write(json.dumps(row))
                    window_loss, window_tokens = 0.0, 0
                if updates % CFG['eval_every'] == 0:
                    last_validation = dict(evaluate(model), evaluated_at_update=updates)
                    history.append(dict(update=updates, validation=last_validation))
                    tqdm.write(json.dumps(dict(update=updates, validation=last_validation)))
                    if last_validation['response_loss'] < best_loss:
                        best_loss = last_validation['response_loss']
                        save_checkpoint('best.pt', model, optimizer, updates, last_validation)
                if updates % CFG['save_every'] == 0:
                    save_checkpoint('latest.pt', model, optimizer, updates, last_validation)
                    save_history(history)
            pair = pairs[pair_index]
            if pair_response_seen == pair['length']-pair['prompt_tokens']:
                pairs_completed += 1
                finished_in_epoch += 1
        if finished_in_epoch == len(train_ids):
            completed_epochs += 1
        if updates >= planned_updates:
            break
final = dict(evaluate(model), evaluated_at_update=updates)
history.append(dict(update=updates, validation=final, final=True))
if final['response_loss'] < best_loss:
    best_loss = final['response_loss']
    save_checkpoint('best.pt', model, optimizer, updates, final)
save_checkpoint('latest.pt', model, optimizer, updates, final)
save_history(history)
torch.cuda.synchronize()
report = dict(initial_validation=initial, final_validation=final, best_response_loss=best_loss,
              updates=updates, train_response_tokens=seen_response_tokens, processed_model_tokens=processed_tokens,
              pairs_started=pairs_started, pairs_completed=pairs_completed, completed_epochs=completed_epochs,
              stop_reason='epochs_completed' if completed_epochs == CFG['epochs'] else 'update_limit',
              training_seconds=training_seconds, response_tokens_per_second=seen_response_tokens/training_seconds,
              wall_seconds=time.perf_counter()-wall_start,
              peak_allocated_MiB=torch.cuda.max_memory_allocated()/2**20,
              peak_reserved_MiB=torch.cuda.max_memory_reserved()/2**20)
print(json.dumps(report, ensure_ascii=False, indent=2))


Response train updates:   0%|          | 0/1000 [00:00<?, ?it/s]

{"update": 1, "epoch": 1, "response_tokens": 3, "train_response_loss": 5.748879909515381, "grad_norm": 3.2901906967163086}
{"update": 50, "epoch": 1, "response_tokens": 376, "train_response_loss": 4.787965242089279, "grad_norm": 2.274834632873535}
{"update": 100, "epoch": 1, "response_tokens": 750, "train_response_loss": 4.139701506670783, "grad_norm": 2.408243417739868}
{"update": 150, "epoch": 1, "response_tokens": 1150, "train_response_loss": 3.74303738117218, "grad_norm": 1.869093418121338}
{"update": 200, "epoch": 1, "response_tokens": 1525, "train_response_loss": 3.6467857666015626, "grad_norm": 3.4878456592559814}
{"update": 200, "validation": {"response_loss": 3.5475674698153, "response_perplexity": 34.728735957253875, "response_accuracy": 0.15664765247915752, "response_tokens": 2279, "pairs": 8, "full_validation": false, "evaluated_at_update": 200}}
{"update": 250, "epoch": 1, "response_tokens": 1897, "train_response_loss": 3.5579852958520255, "grad_norm": 3.3049843311309814}


## 8. 입력을 주고 화자2 답변 생성

reply(model, question)에 화자1의 입력 내용만 전달한다. 학습과 동일한 화자 표기와 BOS를 자동으로 붙이고
**생성한 답변만 반환**한다. EOS 또는 최대 생성 byte 수에서 멈춘다.
잘못된 UTF-8은 �로 표시되며 짧은 학습으로 자연스러운 답변을 보장하지 않는다.

latest.pt는 마지막/주기적 모델, best.pt는 고정 validation 표본의 response loss가 가장 낮은 모델이다.
초기화보다 개선되지 않으면 best는 update 0일 수도 있으며 checkpoint에 update 수를 기록한다.
graph/body IDs/count, NT sign, model/optimizer state, 설정과 데이터 fingerprint를 저장한다.
데이터 중간 위치/hidden state/난수 상태를 저장하는 정확한 중단 재개 기능은 포함하지 않는다.
체크포인트를 후속 학습에 쓰려면 쌍 경계에서 새 데이터 순회를 시작해야 한다.
데이터 셀 실행마다 새 결과 폴더를 만들며 학습 셀만 재실행하면 같은 폴더의 latest/best/history를 갱신한다.


In [16]:
@torch.no_grad()
def reply(m, question, max_new_bytes=180):
    question = ' '.join(question.split())
    if not question:
        raise ValueError('화자1 입력을 비어 있지 않게 주세요.')
    m.eval()
    prefix = '화자1: ' + question + '\n화자2: '
    ids = [BOS] + list(prefix.encode('utf-8'))
    # 긴 질문도 chunk로 읽는다. EOS는 입력 prefix에 붙이지 않는다.
    state = None
    for start in range(0, len(ids), CFG['tbptt']):
        tokens = torch.tensor([ids[start:start+CFG['tbptt']]], dtype=torch.long, device=device)
        logits, state = m(tokens, state)
    generated = []
    for _ in range(max_new_bytes):
        scores = logits[0, -1].clone()
        scores[BOS] = -float('inf')
        token = int(scores.argmax())
        if token == EOS:
            break
        generated.append(token)
        logits, state = m(torch.tensor([[token]], device=device), state)
    return decode(generated)

question = '앴'
answer = reply(model, question)
print('화자1:', question)
print('화자2:', answer)
result_path = RUN_DIR / 'result.json'
result_path.write_text(json.dumps(dict(config=CFG, versions=VERSIONS, sources=source_manifest,
                                      data_config=DATA_CFG, data_spec=data_spec, data_stats=data_stats,
                                      graph=graph_stats, metrics=report, sample=dict(input=question, output=answer)),
                                  ensure_ascii=False, indent=2), encoding='utf-8')
nt_table.to_csv(RUN_DIR / 'neurotransmitters.csv')
print('Drive 결과 폴더:', RUN_DIR)
print('latest.pt / best.pt / history.json / result.json / dataset_manifest.json / neurotransmitters.csv')

# 같은 클래스 정의를 실행한 뒤 저장 모델을 불러오는 예시 (기본 실행 안 함):
# saved = torch.load(RUN_DIR / 'best.pt', map_location='cpu', weights_only=True)
# g = saved['graph']
# loaded = ConnectomeLM(saved['config'], g['src'].numpy(), g['dst'].numpy(), g['counts'].numpy(),
#                      saved['edge_sign'].numpy(), saved['input_nodes'].numpy(), saved['output_nodes'].numpy())
# loaded.load_state_dict(saved['model_state'])
# loaded.to(device).eval()
# print(reply(loaded, '좋아하는 음식이 뭐예요?'))


화자1: 앴
화자2: 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 앴 
Drive 결과 폴더: /content/drive/MyDrive/text_dataset/dataset/flylm_runs/20260907_151606_577848
latest.pt / best.pt / history.json / result.json / dataset_manifest.json / neurotransmitters.csv


## 결과 해석

1. data_stats에서 유효한 2인 대화/입력–답변 쌍 수와 독백/다인 대화 등의 제외 수를 확인한다.
2. 그래프가 8,192개인지, 도달성/NT 적용률 및 수치/response-mask 검사가 통과하는지 확인한다.
3. train_response_loss와 validation.response_loss를 함께 본다. 입력 token을 잘 복사해서 얻는 낮은 loss가 아니다.
   validation은 별도 대화지만 작은 답변 표본이므로 corpus 전체 일반화 성능을 대표하지 않는다.
4. stop_reason=update_limit이면 corpus 전체 학습 완료가 아니다. 전체 epoch는 train_updates=None으로 실행한다.
   UTF-8 byte 단위라 한글 한 글자가 여러 token이며, 8-token TBPTT는 매우 짧은 gradient 문맥이다.
5. 모든 뉴런/edge에 gradient가 생기는지는 graph 경로와 구조 검사 출력으로 확인한다.
   MaleCNS topology의 효과는 같은 데이터/예산의 일반 RNN 및 degree 보존 random graph와 별도 비교해야 한다.

출처:
- [MaleCNS 공식 다운로드](https://male-cns.janelia.org/download/) — FlyEM/Janelia 및 공동 연구팀의 v1.0 데이터, CC-BY.
- [Network statistics of the whole-brain connectome of Drosophila](https://www.nature.com/articles/s41586-024-07968-y)
  — 전달물질 부호 추정의 맥락. 다른 개체의 자료이며 MaleCNS prediction 정확도의 근거가 아니다.
- [PyTorch checkpoint](https://docs.pytorch.org/docs/stable/checkpoint.html),
  [Arrow RecordBatchFileReader](https://arrow.apache.org/docs/python/generated/pyarrow.ipc.RecordBatchFileReader.html),
  [SciPy directed BFS](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csgraph.breadth_first_order.html),
  [Colab Drive I/O](https://research.google.com/colaboratory/faq.html).
